In [1]:
print("Hello")

Hello


In [2]:
import nest_asyncio
nest_asyncio.apply()

import random
from pydantic_ai import Agent, RunContext
from dotenv import load_dotenv
from dataclasses import dataclass

load_dotenv() 

True

In [3]:
# Simple tool custom

game_agent=Agent(
    "groq:moonshotai/kimi-k2-instruct-0905",
    system_prompt="you are a helpful game master assistant.",
)

@game_agent.tool_plain
def roll_dice(sides:int) -> int:
    """
    Use this to roll a dice with a specific number of sides.
    """
    print(f"\n[System] Tool Used! Rolling a {sides}-sided dice ...")
    return random.randint(1, sides)

res = game_agent.run_sync("I am attacking a dragon! can you roll a 20-sided dice for me?")
print("\n Agent: ", res.output)


[System] Tool Used! Rolling a 20-sided dice ...

 Agent:  You rolled a **17**! That's a solid hit—let's see if it's enough to wound the dragon.


In [7]:
# context Tools

# Define what secret data we want to pass
@dataclass
class CustomerDatabase:
    company_name: str
    secret_discounts: dict

# tell the agent to expect this type of data
store_agent=Agent(
    "groq:llama-3.3-70b-versatile",
    deps_type=CustomerDatabase,
    system_prompt="you are a store assistant. Check for discounts if asked."
)

# Access our injected data through "ctx"
@store_agent.tool
def check_discounts(ctx: RunContext[CustomerDatabase], item_name:str) -> str:
    """ 
    Use this to check if a specific item has a discount avilable.
    """
    print(f"\n [System] Checking the {ctx.deps.company_name} DB for {item_name}")
    
    discount=ctx.deps.secret_discounts.get(item_name.lower(), "0%")
    return(f"discount is {discount}.")

# Provide the secure data at runtime
my_live_db = CustomerDatabase(
    company_name="Apple",
    secret_discounts={"laptop": "10%", "mouse":"5%"}
)

res=store_agent.run_sync("Do you have any discounts on laptop", deps=my_live_db)
print("\n Agent: ", res.output)


 [System] Checking the Apple DB for laptop

 Agent:  We currently have a 10% discount available on laptops. Would you like to know more about our laptop models or proceed with a purchase?
